In [1]:
%pip install transformers
%pip install accelerate
%pip install emoji wordsegment
%pip install huggingface_hub
%pip install torch --index-url https://download.pytorch.org/whl/cu124
%pip install pandas scikit-learn
%pip install ftfy regex tqdm
%pip install PyEvALL

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import pandas as pd
import numpy as np
import re
import os
import torch.nn as nn
import torch.nn.functional as F
import emoji
from wordsegment import segment
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import (
    TrainingArguments,
    Trainer,
    XLMRobertaTokenizer,
    XLMRobertaModel,
    EarlyStoppingCallback,
    CLIPProcessor,
    CLIPVisionModel
)
import json
import os
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
import tempfile

In [ ]:
import os
os.chdir("/home/a67606/Thesis/EXIST_MEMES")

/home/a67606
total 104
drwxr-x--- 17 a67606 a67606  4096 Aug  1 16:56 .
drwxr-xr-x  4 root   root    4096 Mar 16 10:35 ..
-rw-------  1 a67606 a67606 13942 Aug  1 17:19 .bash_history
-rw-r--r--  1 a67606 a67606   220 Mar 31  2024 .bash_logout
-rw-r--r--  1 a67606 a67606  3771 Mar 31  2024 .bashrc
drwx------  8 a67606 a67606  4096 May 13 19:52 .cache
drwxrwxr-x  3 a67606 a67606  4096 May 13 19:52 .config
drwx------  3 a67606 a67606  4096 May 12 18:05 .copilot
drwxrwxr-x  3 a67606 a67606  4096 May 12 18:11 .dotnet
drwxrwxr-x  2 a67606 a67606  4096 Aug  1 11:22 .ipynb_checkpoints
drwxrwxr-x  3 a67606 a67606  4096 Apr 22 13:24 .ipython
drwxrwxr-x  3 a67606 a67606  4096 Apr 26 16:58 .jupyter
drwxrwxr-x  7 a67606 a67606  4096 Apr 22 13:35 jupyter-env
drwx------  4 a67606 a67606  4096 Apr 22 13:20 .local
drwx------  3 a67606 a67606  4096 Apr 22 13:38 .nv
-rw-r--r--  1 a67606 a67606   807 Mar 31  2024 .profile
-rw-------  1 a67606 a67606    12 Apr 22 13:14 .python_history
drwxrwxr-x  3 a67606 

In [3]:
raw_data = pd.read_json('./Dataset/memes/training/EXIST2026_training.json')
raw_data = raw_data.transpose()
raw_data.head()

FileNotFoundError: File ./Dataset/memes/training/EXIST2026_training.json does not exist

In [ ]:
def has_modality(row, modality):
    sens = row.get('sensorial')
    if sens is None or not isinstance(sens, dict):
        return False
    by_user = sens.get('modalities', {}).get(modality, {}).get('by_user', {})
    return len(by_user) > 0

print(f"Total samples: {len(raw_data)}")
print(f"With any sensorial: {raw_data.apply(lambda r: r.get('sensorial') is not None, axis=1).sum()}")
for mod in ['ET', 'HR', 'EEG']:
    n = raw_data.apply(lambda r: has_modality(r, mod), axis=1).sum()
    print(f"With {mod}: {n}")

# Subject counts
sub_counts = raw_data.apply(
    lambda r: len(r.get('sensorial', {}).get('users', [])) if r.get('sensorial') else 0,
    axis=1
)
print(f"\nSubjects per sample: mean={sub_counts.mean():.2f}, max={sub_counts.max()}, min={sub_counts[sub_counts > 0].min() if (sub_counts > 0).any() else 0}")
print(f"Distribution:\n{sub_counts.value_counts().sort_index()}")

Total samples: 3984
With any sensorial: 3984
With ET: 3984
With HR: 3984
With EEG: 3984

Subjects per sample: mean=3.88, max=4, min=2
Distribution:
2      18
3     423
4    3543
Name: count, dtype: int64


# Remove unecessary columns

In [ ]:
dataset = raw_data.copy()

In [ ]:
dataset.drop(['id_EXIST', 'number_annotators', 'annotators', 'lang', 'split'], axis=1, inplace=True)
dataset.head()

,text,meme,path_memes,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3
110887,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,memes/110887.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Multiracial, White or Cau...","[High school degree or equivalent, Master’s de...","[Sweden, United States, Spain, Portugal, Spain...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, DIRECT, DIRECT, DIRECT, DIRECT, ...","[[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION], [M..."
110466,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,memes/110466.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, White or Caucasian, Hispa...","[Bachelor’s degree, High school degree or equi...","[Spain, Portugal, Spain, Canada, United States...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, NO, YES, YES, NO, NO]","[DIRECT, -, DIRECT, DIRECT, -, -]","[[STEREOTYPING-DOMINANCE], [-], [STEREOTYPING-..."
111269,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,memes/111269.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, other,...","[Bachelor’s degree, Master’s degree, High scho...","[Spain, Spain, Spain, Italy, Spain, Portugal]","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, DIRECT, -]","[[OBJECTIFICATION, SEXUAL-VIOLENCE], [OBJECTIF..."
110593,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,memes/110593.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[High school degree or equivalent, Master’s de...","[Poland, Portugal, Spain, United Kingdom, Mexi...","{'users': ['ES1', 'ES3'], 'modalities': {'ET':...","[NO, YES, NO, NO, NO, NO]","[-, JUDGEMENTAL, -, -, -, -]","[[-], [STEREOTYPING-DOMINANCE, OBJECTIFICATION..."
110946,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,memes/110946.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[Bachelor’s degree, Bachelor’s degree, Bachelo...","[Portugal, Chile, Italy, Portugal, Spain, Chile]","{'users': ['ES1', 'ES2', 'ES3', 'ES4'], 'modal...","[YES, YES, YES, YES, NO, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, -, -]","[[OBJECTIFICATION], [SEXUAL-VIOLENCE], [OBJECT..."


In [ ]:
dataset.describe()

,text,meme,path_memes,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3
count,3984,3984,3984,3984,3984,3984,3984,3984,3984,3984,3984,3984
unique,3981,3984,3984,1,2,111,125,151,3984,64,785,2889
top,"Para las mujeres: ""¡Hola hermosa!"" Halago Para...",211643.jpeg,memes/211643.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 18-22, 23-45, 46+]","[White or Caucasian, White or Caucasian, White...","[High school degree or equivalent, Bachelor’s ...","[Hungary, Chile, United Kingdom, Australia, It...","{'users': ['EN1', 'EN2', 'EN3', 'EN5'], 'modal...","[YES, YES, YES, YES, YES, YES]","[-, -, -, -, -, -]","[[-], [-], [-], [-], [-], [-]]"
freq,2,1,1,3984,2005,161,81,27,1,572,373,373


In [ ]:
for i in range(3):
    path = os.path.join('./Dataset/memes/training/', dataset['path_memes'].iloc[i])
    exists = os.path.exists(path)
    print(f"{path} → {'✓ exists' if exists else '✗ MISSING'}")

./Dataset/memes/training/memes/110887.jpeg → ✓ exists
./Dataset/memes/training/memes/110466.jpeg → ✓ exists
./Dataset/memes/training/memes/111269.jpeg → ✓ exists


In [ ]:
LABELS_2_1 = ['YES', 'NO']
LABELS_2_2 = ['NONE', 'JUDGEMENTAL', 'DIRECT']
LABELS_2_3 = ['NONE', 'IDEOLOGICAL-INEQUALITY', 'MISOGYNY-NON-SEXUAL-VIOLENCE',
               'SEXUAL-VIOLENCE', 'OBJECTIFICATION', 'STEREOTYPING-DOMINANCE']

THRESHOLD_GRID = np.arange(0.05, 0.65, 0.05).tolist()


TASK_2_3_HIERARCHY = {
    "YES": ["IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION",
            "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE"],
    "NO": []
}

DEMO_ETHNICITY_VALS   = sorted(['Asian', 'Black or African American', 'Hispano or Latino',
                                 'Middle Eastern', 'Multiracial', 'White or Caucasian', 'other'])
DEMO_STUDY_VALS       = sorted(['Bachelor\'s degree', 'Doctorate',
                                 'High school degree or equivalent',
                                 'Less than high school diploma', 'Master\'s degree', 'other'])
DEMO_COUNTRY_VALS     = sorted(['Afghanistan', 'Argentina', 'Australia', 'Austria', 'Azerbaijan',
                                 'Bangladesh', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile',
                                 'China', 'Colombia', 'Croatia', 'Cuba', 'Czech Republic',
                                 'Dominican Republic', 'Ecuador', 'Estonia', 'Finland', 'France',
                                 'Germany', 'Greece', 'Hungary', 'India', 'Ireland', 'Israel',
                                 'Italy', 'Latvia', 'Mexico', 'Morocco', 'Netherlands',
                                 'New Zealand', 'Nicaragua', 'Nigeria', 'Peru', 'Poland',
                                 'Portugal', 'Romania', 'Russian Federation', 'Serbia', 'Slovenia',
                                 'South Africa', 'Spain', 'Sweden', 'Turkey', 'United Arab Emirates',
                                 'United Kingdom', 'United States', 'Uruguay', 'Venezuela',
                                 'Viet Nam', 'Zimbabwe'])

DEMO_FEATURE_GROUPS = [
    ('ethnicities_annotators',  DEMO_ETHNICITY_VALS),
    ('study_levels_annotators', DEMO_STUDY_VALS),
    ('countries_annotators',    DEMO_COUNTRY_VALS),
]

DEMO_VEC_DIM    = len(DEMO_ETHNICITY_VALS) + len(DEMO_STUDY_VALS) + len(DEMO_COUNTRY_VALS)  # 66
print(f"Annotator vector dim: {DEMO_VEC_DIM}")

Annotator vector dim: 66


# Utils

In [ ]:
def remap_labels_flat(annotations):
    """For tasks where each annotator gives a single label.
    - Convert '-' to 'NONE' (means 'not sexist')
    - Filter out 'UNKNOWN' (means 'annotator didn't decide')
    """
    return [
        "NONE" if label == "-" else label
        for label in annotations
        if label != "UNKNOWN"
    ]

def remap_labels_nested(annotations):
    """For tasks where each annotator gives multiple labels.
    - Convert '-' to 'NONE'
    - Filter out 'UNKNOWN' from each annotator's list
    - Skip annotators whose list becomes empty
    """
    result = []
    for annotator in annotations:
        cleaned = [
            "NONE" if label == "-" else label
            for label in annotator
            if label != "UNKNOWN"
        ]
        if cleaned:  # only keep annotators with valid labels
            result.append(cleaned)
    return result

In [ ]:
def labels_to_id(label_list):
    result = {}
    for i, label in enumerate(label_list):
        result[label] = i
    return result

In [ ]:
def id_to_label(label_to_id):
    result = {}
    for label, i in label_to_id.items():
        result[i] = label
    return result

In [ ]:
def extract_features(text):
    censored_words = re.findall(r'\b\w*\*+\w*\b', text)

    return {
        "contains_censored": int(len(censored_words) > 0),
        "censored_count": len(censored_words)
    }

In [ ]:
def replace_user_mentions(text: str) -> str:
    pattern = r'@\w+'
    return re.sub(pattern, '<USER_MENTION>', text)

In [ ]:
def replace_urls(text: str) -> str:
  text = re.sub(r'http\S+|www\S+', '<URL>', text)
  return text

In [ ]:
def normalize_hashtags(text):

    def process_hashtag(match):
        tag = match.group()[1:]

        # underscore hashtags
        if "_" in tag:
            words = tag.split("_")

        # camel case
        elif re.search(r'[A-Z]', tag):
            words = re.sub(r'(?<!^)(?=[A-Z])', ' ', tag).split()

        else:
            try:
                words = segment(tag)
            except ValueError:
                words = [tag]  # fallback

        return " ".join(words)

    return re.sub(r'#\w+', process_hashtag, text)

In [ ]:
def convert_emojis_to_text(text: str) -> str:
    text = emoji.demojize(text)
    #text = text.replace(":", " ").replace("_", " ")
    return text

In [ ]:
def remove_random_chars_and_spaces(text: str) -> str:
  text = re.sub(r'[^\w\s<>]', ' ', text)
  return re.sub(r'\s+', ' ', text).strip()


In [ ]:
def limit_token_repetition(text, max_repeats=2):
    words = text.split()
    result = []

    prev = None
    count = 0

    for w in words:
        if w == prev:
            count += 1
        else:
            prev = w
            count = 1

        if count <= max_repeats:
            result.append(w)

    return " ".join(result)

In [ ]:
def replace_profanity(text: str) -> str:
    profanity_pattern = r'\b[a-zA-Z]*[*!@#$%^&]+[a-zA-Z]*\b'
    return re.sub(profanity_pattern, "<PROFANITY>", text)

In [ ]:
def pre_process_text(text: str) -> str:
    return remove_random_chars_and_spaces(limit_token_repetition(replace_profanity(
           replace_user_mentions(
            convert_emojis_to_text(
                replace_urls(
                    normalize_hashtags(text).lower()
                )
            )
      )
    )))

In [ ]:
def divide_dict_values(d: dict, divisor: float) -> dict:
    return {k: v / divisor for k, v in d.items()}

In [ ]:
def get_soft_distribution(label_series, available_labels):
    if not label_series:
        return None
    
    counts = {label: 0 for label in available_labels}
    length = len(label_series)
    for label in label_series:
        if label in counts:
            counts[label] += 1
    counts = divide_dict_values(counts, length)
    return list(counts.values())


In [ ]:
def annotations_matrix_to_distribution(annotations, labels):
    if not annotations:
        return None
    
    counts = {label: 0 for label in labels}
    num_annotators = len(annotations)

    for annotator in annotations:
        for label in annotator:
            if label in counts:
                counts[label] += 1

    return [counts[label] / num_annotators for label in labels]

In [ ]:
def compute_annotator_vector(row, feature_groups):
    """
    GrootWatch-style annotator demographic vector.
    For each feature group: count annotators per category → concatenate → normalize.
    Normalization: divide by total annotator count (so vector sums to num_features=3,
    or alternatively to 1 across the full vector).
    """
    parts = []
    for col, vocab in feature_groups:
        vocab_index = {v: i for i, v in enumerate(vocab)}
        counts = np.zeros(len(vocab), dtype=np.float32)
        cell = row.get(col, None)
        if isinstance(cell, list):
            for val in cell:
                if val is not None:
                    idx = vocab_index.get(str(val).strip(), None)
                    if idx is not None:
                        counts[idx] += 1.0
        parts.append(counts)

    vec = np.concatenate(parts)  # [66]

    # Sum-normalize (matches GrootWatch's "[1,1,1,2,1,1,2] → [0.1,0.1,0.1,0.2,0.1,0.1,0.2]")
    total = vec.sum()
    if total > 1e-9:
        vec = vec / total

    return vec

In [ ]:
def build_vocab(series, feature_name):
    """Collect all unique values across all annotators for one feature."""
    values = set()
    for cell in series:
        if isinstance(cell, list):
            for v in cell:
                if v is not None and str(v).strip():
                    values.add(str(v).strip())
    vocab = {v: i + 1 for i, v in enumerate(sorted(values))}  # 0 = padding/unknown
    print(f"  {feature_name}: {len(vocab)} unique values → {sorted(vocab.keys())}")
    return vocab

In [ ]:
def encode_demo_vector(row, vocabs, features, num_annotators=6):
    """
    Build a [num_annotators, num_features] integer index matrix for one sample.
    Each annotator × feature → vocab index (0 if missing/unknown).
    Returns a list of lists: shape [num_annotators, num_features].
    """
    result = []
    for ann_idx in range(num_annotators):
        ann_vec = []
        for feat in features:
            cell = row.get(feat, None)
            if isinstance(cell, list) and ann_idx < len(cell):
                val = str(cell[ann_idx]).strip() if cell[ann_idx] is not None else ''
                ann_vec.append(vocabs[feat].get(val, 0))
            else:
                ann_vec.append(0) 
        result.append(ann_vec)
    return result

In [ ]:
TASK_2_2_HIERARCHY = {
    "YES": ["DIRECT", "JUDGEMENTAL"],
    "NO": []
}

TASK_2_3_HIERARCHY = {
    "YES": [
        "IDEOLOGICAL-INEQUALITY",
        "STEREOTYPING-DOMINANCE",
        "OBJECTIFICATION",
        "SEXUAL-VIOLENCE",
        "MISOGYNY-NON-SEXUAL-VIOLENCE"
    ],
    "NO": []
}

In [ ]:
def _extract_metrics_from_report(report, metric_list):
    """Pull metric scores out of a PyEvALL report object."""
    results = {}
    if not hasattr(report, 'report') or 'metrics' not in report.report:
        return results
    
    metrics_dict = report.report['metrics']
    for metric in metric_list:
        if metric not in metrics_dict:
            continue
        if metrics_dict[metric].get('status') != 'OK':
            continue
        try:
            results[metric] = float(
                metrics_dict[metric]['results']['average_per_test_case']
            )
        except (KeyError, TypeError, ValueError):
            pass
    return results

def _format_for_pyevall(probs, sample_ids, task, evaluation, thresholds_c=None):
    """Convert probability arrays to PyEvALL JSON format.
    
    thresholds_c: optional list of 6 floats for Task C hard mode.
                  If None, defaults to 0.5 for all classes.
    """
    out = []
    for i, sid in enumerate(sample_ids):
        entry = {"test_case": "EXIST2025", "id": str(sid)}

        if task == '2_1':
            p_yes = float(probs[i][0])
            p_no  = float(probs[i][1])
            if evaluation == 'hard':
                entry["value"] = "YES" if p_yes >= p_no else "NO"
            else:
                entry["value"] = {"YES": p_yes, "NO": p_no}

        elif task == '2_2':
            label_map = {"NONE": "NO", "JUDGEMENTAL": "JUDGEMENTAL", "DIRECT": "DIRECT"}
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_2)}
            if evaluation == 'hard':
                entry["value"] = max(relevant, key=relevant.get)
            else:
                entry["value"] = relevant

        elif task == '2_3':
            label_map = {"NONE": "NO"}
            label_map.update({lbl: lbl for lbl in LABELS_2_3[1:]})
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_3)}

            if evaluation == 'hard':
                # Use per-class thresholds if provided, else 0.5
                threshs = thresholds_c if thresholds_c is not None else [0.5] * len(LABELS_2_3)
                # Map thresholds to the remapped label names
                thresh_map = {label_map[lbl]: threshs[idx] for idx, lbl in enumerate(LABELS_2_3)}
                hard_labels = [lbl for lbl, p in relevant.items() if p >= thresh_map[lbl]]
                if not hard_labels:
                    hard_labels = [max(relevant, key=relevant.get)]
                entry["value"] = hard_labels
            else:
                entry["value"] = relevant

        out.append(entry)
    return out


def _run_pyevall(probs, gold_dist, sample_ids, task, evaluation,
                 temp_dir='./pyevall_temp_train', thresholds_c=None):
    """Single PyEvALL run for one task and evaluation mode.
    
    thresholds_c: per-class thresholds for Task C hard mode (list of 6 floats).
                  Ignored for all other tasks and for soft evaluation.
    """
    os.makedirs(temp_dir, exist_ok=True)

    preds = _format_for_pyevall(probs, sample_ids, task, evaluation,
                                 thresholds_c=thresholds_c)
    gold  = _format_for_pyevall(gold_dist, sample_ids, task, evaluation)
    # Note: thresholds_c is intentionally NOT passed for gold — gold uses raw
    # soft distributions and PyEvALL binarizes them internally for hard mode.

    pred_path = f'{temp_dir}/pred_{task}_{evaluation}.json'
    gold_path = f'{temp_dir}/gold_{task}_{evaluation}.json'
    with open(pred_path, 'w') as f: json.dump(preds, f)
    with open(gold_path, 'w') as f: json.dump(gold, f)

    if evaluation == 'hard':
        metric_list = ['ICM', 'ICMNorm', 'FMeasure']
    else:
        if task == '2_3':
            metric_list = ['ICMSoft', 'ICMSoftNorm']
        else:
            metric_list = ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']

    params = {}
    if task == '2_2':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_2_HIERARCHY
    elif task == '2_3':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_3_HIERARCHY

    try:
        evaluator = PyEvALLEvaluation()
        report    = evaluator.evaluate(pred_path, gold_path, metric_list, **params)
        return _extract_metrics_from_report(report, metric_list)
    except Exception as e:
        print(f"  ! PyEvALL failed for task {task} ({evaluation}): {e}")
        return {}


def _search_thresholds_task_c(probs_c, targets_c, ids_c, temp_dir):
    """Greedy coordinate search over per-class thresholds for Task C hard ICMNorm.
    
    Optimizes one class at a time holding the others fixed.
    12 candidates × 6 classes = 72 PyEvALL calls per epoch.

    Returns: (best_thresholds: list[float], final_scores: dict)
    """
    best_thresholds = [0.5] * len(LABELS_2_3)

    for class_idx in range(len(LABELS_2_3)):
        best_t   = best_thresholds[class_idx]
        best_icm = -np.inf

        for t in THRESHOLD_GRID:
            candidate = best_thresholds.copy()
            candidate[class_idx] = float(t)

            scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                  temp_dir=temp_dir, thresholds_c=candidate)
            icm = scores.get('ICMNorm', -np.inf)
            if icm > best_icm:
                best_icm = icm
                best_t   = float(t)

        best_thresholds[class_idx] = best_t

    # Final eval with all thresholds locked in
    final_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                temp_dir=temp_dir, thresholds_c=best_thresholds)
    return best_thresholds, final_scores


# === Main compute_metrics function ===

def compute_metrics(eval_pred):
    """Compute all metrics using official PyEvALL library.

    Runs 6 evaluations (3 tasks × {hard, soft}) plus a per-class threshold
    search for Task C hard (72 extra PyEvALL calls via greedy coordinate search).
    """
    logits_a, logits_b, logits_c = eval_pred.predictions
    labels_a, labels_b, labels_c = eval_pred.label_ids
    logits_a = np.asarray(logits_a)
    logits_b = np.asarray(logits_b)
    logits_c = np.asarray(logits_c)
    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    labels_c = np.asarray(labels_c)
    metrics = {}

    with tempfile.TemporaryDirectory() as temp_dir:

        # === Task A ===
        if labels_a.ndim == 2:
            labels_a = labels_a[:, 1]

        mask_a = labels_a != -100.0
        if mask_a.any():
            prob_no  = 1 / (1 + np.exp(-logits_a[mask_a].squeeze(-1)))
            probs_a  = np.stack([1 - prob_no, prob_no], axis=-1)
            targets_a = np.stack([1 - labels_a[mask_a], labels_a[mask_a]], axis=-1)
            ids_a    = [str(i) for i in range(len(probs_a))]

            hard_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'hard',
                                       temp_dir=temp_dir)
            for k, v in hard_scores.items():
                metrics[f'task_a_{k.lower()}_hard'] = v

            soft_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_a_{k.lower()}_soft'] = v

        # === Task B ===
        mask_b = labels_b[:, 0] != -1.0
        if mask_b.any():
            x        = logits_b[mask_b]
            x        = x - x.max(axis=-1, keepdims=True)
            probs_b  = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
            targets_b = labels_b[mask_b]
            ids_b    = [str(i) for i in range(len(probs_b))]

            hard_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'hard',
                                       temp_dir=temp_dir)
            for k, v in hard_scores.items():
                metrics[f'task_b_{k.lower()}_hard'] = v

            soft_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_b_{k.lower()}_soft'] = v

        # === Task C ===
        mask_c = labels_c[:, 0] != -1.0
        if mask_c.any():
            probs_c  = 1 / (1 + np.exp(-logits_c[mask_c]))
            targets_c = labels_c[mask_c]
            ids_c    = [str(i) for i in range(len(probs_c))]

            # Soft (unchanged)
            soft_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_c_{k.lower()}_soft'] = v

            # Hard — default threshold (0.5) for reference
            hard_scores_default = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                               temp_dir=temp_dir)
            for k, v in hard_scores_default.items():
                metrics[f'task_c_{k.lower()}_hard_default'] = v

            # Hard — tuned thresholds via greedy coordinate search
            best_thresholds, hard_scores_tuned = _search_thresholds_task_c(
                probs_c, targets_c, ids_c, temp_dir
            )
            for k, v in hard_scores_tuned.items():
                metrics[f'task_c_{k.lower()}_hard'] = v  # this becomes the canonical hard score

            # Log per-class thresholds so you can watch them evolve
            for j, label in enumerate(LABELS_2_3):
                metrics[f'task_c_thresh_{label.lower().replace("-", "_")}'] = round(best_thresholds[j], 2)

    return metrics

In [ ]:
print(annotations_matrix_to_distribution([
      ["IDEOLOGICAL-INEQUALITY", "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE"],
      ["SEXUAL-VIOLENCE"],
      ["NONE"],
      ["IDEOLOGICAL-INEQUALITY", "SEXUAL-VIOLENCE"],
      ["MISOGYNY-NON-SEXUAL-VIOLENCE"],
      ["SEXUAL-VIOLENCE"]
    ], LABELS_2_3))

[0.16666666666666666, 0.3333333333333333, 0.3333333333333333, 0.6666666666666666, 0.0, 0.0]


In [ ]:
# Get all unique values per feature to know vector dimensions
for col in ['ethnicities_annotators', 'study_levels_annotators', 'countries_annotators']:
    all_vals = set()
    for cell in raw_data[col]:
        if isinstance(cell, list):
            for v in cell:
                if v is not None:
                    all_vals.add(str(v).strip())
    print(f"{col}: {len(all_vals)} unique values → {sorted(all_vals)}")

ethnicities_annotators: 7 unique values → ['Asian', 'Black or African American', 'Hispano or Latino', 'Middle Eastern', 'Multiracial', 'White or Caucasian', 'other']
study_levels_annotators: 6 unique values → ['Bachelor’s degree', 'Doctorate', 'High school degree or equivalent', 'Less than high school diploma', 'Master’s degree', 'other']
countries_annotators: 53 unique values → ['Afghanistan', 'Argentina', 'Australia', 'Austria', 'Azerbaijan', 'Bangladesh', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cuba', 'Czech Republic', 'Dominican Republic', 'Ecuador', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'India', 'Ireland', 'Israel', 'Italy', 'Latvia', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Nicaragua', 'Nigeria', 'Peru', 'Poland', 'Portugal', 'Romania', 'Russian Federation', 'Serbia', 'Slovenia', 'South Africa', 'Spain', 'Sweden', 'Turkey', 'United Arab Emirates', 'United Kingdom', 'United States', 'Uruguay', 'Venez

In [ ]:
print(get_soft_distribution(['NONE', 'NONE', 'NONE', 'NONE', 'DIRECT', 'NONE'], LABELS_2_2))

[0.8333333333333334, 0.0, 0.16666666666666666]


In [ ]:
print(annotations_matrix_to_distribution([
      ["STEREOTYPING-DOMINANCE", "OBJECTIFICATION", "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE"],
      ["SEXUAL-VIOLENCE"],
      ["NONE"],
      ["OBJECTIFICATION"],
      ["NONE"],
      ["NONE"]
    ], LABELS_2_3))

[0.5, 0.0, 0.16666666666666666, 0.3333333333333333, 0.3333333333333333, 0.16666666666666666]


In [ ]:
print(pre_process_text('@ShahidForChange @TeamPelosi @randomUser Quiet, you f*cking sexist ^%$#.https://t.co/6GqlfIKeE3 Los mejores 100 #videojuegos de la #Historia 😀😀😀😀 #gamers #GamerGate https://t.co/r6KoQwNiwh'))

<USER_MENTION> <USER_MENTION> quiet you <PROFANITY> sexist <URL> los mejores 100 videojuegos de la historia grinning_face grinning_face grinning_face grinning_face gamers gamer gate <URL>


In [ ]:
dataset['text_preprocessed'] = dataset['text'].apply(pre_process_text)
dataset['labels_task2_1_soft'] = dataset['labels_task2_1'].apply(lambda x: get_soft_distribution(x, LABELS_2_1))
dataset['labels_task2_2'] = dataset['labels_task2_2'].apply(lambda x: remap_labels_flat(x))
dataset['labels_task2_2_soft'] = dataset['labels_task2_2'].apply(lambda x: get_soft_distribution(x, LABELS_2_2))
dataset['labels_task2_3'] = dataset['labels_task2_3'].apply(lambda x: remap_labels_nested(x))
dataset['labels_task2_3_soft'] = dataset['labels_task2_3'].apply(lambda x: annotations_matrix_to_distribution(x, LABELS_2_3))
dataset['demo_vec'] = dataset.apply(
    lambda row: compute_annotator_vector(row, DEMO_FEATURE_GROUPS), axis=1
)
print(f"Before filtering: {len(dataset)} samples")
dataset = dataset[dataset['labels_task2_1_soft'].notna()].reset_index(drop=True)
print(f"After filtering Task A: {len(dataset)} samples")

Before filtering: 3984 samples
After filtering Task A: 3984 samples


In [ ]:
dataset.head()

,text,meme,path_memes,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3,text_preprocessed,labels_task2_1_soft,labels_task2_2_soft,labels_task2_3_soft,demo_vec
0,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,memes/110887.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Multiracial, White or Cau...","[High school degree or equivalent, Master’s de...","[Sweden, United States, Spain, Portugal, Spain...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, DIRECT, DIRECT, DIRECT, DIRECT, ...","[[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION], [M...",a veces quisierair alzumba <URL> <URL> 130 per...,"[1.0, 0.0]","[0.0, 0.16666666666666666, 0.8333333333333334]","[0.0, 0.16666666666666666, 0.5, 0.0, 0.6666666...","[0.0, 0.0, 0.13333334, 0.0, 0.06666667, 0.2, 0..."
1,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,memes/110466.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, White or Caucasian, Hispa...","[Bachelor’s degree, High school degree or equi...","[Spain, Portugal, Spain, Canada, United States...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, NO, YES, YES, NO, NO]","[DIRECT, NONE, DIRECT, DIRECT, NONE, NONE]","[[STEREOTYPING-DOMINANCE], [NONE], [STEREOTYPI...",se necesita cuidadora para adulto mayor fotos ...,"[0.5, 0.5]","[0.5, 0.0, 0.5]","[0.5, 0.16666666666666666, 0.16666666666666666...","[0.0, 0.0, 0.2, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, ..."
2,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,memes/111269.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, other,...","[Bachelor’s degree, Master’s degree, High scho...","[Spain, Spain, Spain, Italy, Spain, Portugal]","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, DIRECT, NONE]","[[OBJECTIFICATION, SEXUAL-VIOLENCE], [OBJECTIF...",tomboy como son el anime y manga pre to tomboy...,"[0.8333333333333334, 0.16666666666666666]","[0.16666666666666666, 0.0, 0.8333333333333334]","[0.16666666666666666, 0.0, 0.16666666666666666...","[0.0, 0.0, 0.13333334, 0.0, 0.0, 0.2, 0.066666..."
3,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,memes/110593.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[High school degree or equivalent, Master’s de...","[Poland, Portugal, Spain, United Kingdom, Mexi...","{'users': ['ES1', 'ES3'], 'modalities': {'ET':...","[NO, YES, NO, NO, NO, NO]","[NONE, JUDGEMENTAL, NONE, NONE, NONE, NONE]","[[NONE], [STEREOTYPING-DOMINANCE, OBJECTIFICAT...",hoy quiero felicitar a todas las mujeres de es...,"[0.16666666666666666, 0.8333333333333334]","[0.8333333333333334, 0.16666666666666666, 0.0]","[0.8333333333333334, 0.0, 0.0, 0.0, 0.16666666...","[0.0, 0.0, 0.21428572, 0.0, 0.0, 0.21428572, 0..."
4,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,memes/110946.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[Bachelor’s degree, Bachelor’s degree, Bachelo...","[Portugal, Chile, Italy, Portugal, Spain, Chile]","{'users': ['ES1', 'ES2', 'ES3', 'ES4'], 'modal...","[YES, YES, YES, YES, NO, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, NONE, NONE]","[[OBJECTIFICATION], [SEXUAL-VIOLENCE], [OBJECT...",duchate guarra gracias memegenerator es,"[0.6666666666666666, 0.3333333333333333]","[0.3333333333333333, 0.0, 0.6666666666666666]","[0.3333333333333333, 0.0, 0.0, 0.1666666666666...","[0.0, 0.0, 0.16666667, 0.0, 0.0, 0.33333334, 0..."


In [ ]:
dataset['text'].tail()

3979    RAMZPAUL @ramzpaul Kristen Dunst wears black t...
3980    Girls: OMG I don't know what to wear today Boy...
3981    When someone asks what my plans for the weeken...
3982    GETS REDDIT GOLD CALLS ME GOLD-DIGGER imgflip....
3983    What people think having a step-sister looks l...
Name: text, dtype: object

# Dataset preparation

In [ ]:
class SexismDataset(Dataset):
    def __init__(self, dataframe, tokenizer, clip_processor, max_length, image_dir):
        self.tokenizer      = tokenizer
        self.clip_processor = clip_processor
        self.max_length     = max_length
        self.image_dir      = image_dir

        self.text         = dataframe['text_preprocessed'].values
        self.image_paths  = dataframe['path_memes'].values
        self.labels_task1 = dataframe['labels_task2_1_soft'].values
        self.labels_task2 = dataframe['labels_task2_2_soft'].values
        self.labels_task3 = dataframe['labels_task2_3_soft'].values
        self.demo_vecs    = dataframe['demo_vec'].values  # [N,] of np.ndarray [66]

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        tokenized = self.tokenizer(
            self.text[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        image_path = os.path.join(self.image_dir, self.image_paths[idx])
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            print("Image not found", image_path)
            image = Image.new('RGB', (224, 224), color=0)

        image_inputs = self.clip_processor(images=image, return_tensors='pt')

        batch = {
            'input_ids':      tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'pixel_values':   image_inputs['pixel_values'].squeeze(0),
            'demo_vec':       torch.tensor(self.demo_vecs[idx], dtype=torch.float32),  # [66]
            'labels_a':       torch.tensor(self.labels_task1[idx], dtype=torch.float32),
        }

        labels_task2 = self.labels_task2[idx]
        if labels_task2 is None:
            batch['labels_b'] = torch.full((len(LABELS_2_2),), -1.0, dtype=torch.float32)
        else:
            labels_task2 = np.array(labels_task2, dtype=np.float32)
            if labels_task2[0] != -1.0:
                batch['labels_b'] = torch.tensor(labels_task2, dtype=torch.float32)
            else:
                batch['labels_b'] = torch.full((len(labels_task2),), -1.0, dtype=torch.float32)

        labels_task3 = self.labels_task3[idx]
        if labels_task3 is None:
            batch['labels_c'] = torch.full((len(LABELS_2_3),), -1.0, dtype=torch.float32)
        else:
            labels_task3 = np.array(labels_task3, dtype=np.float32)
            if labels_task3[0] != -1.0:
                batch['labels_c'] = torch.tensor(labels_task3, dtype=torch.float32)
            else:
                batch['labels_c'] = torch.full((len(labels_task3),), -1.0, dtype=torch.float32)

        return batch

In [ ]:
class MultiHeadModel(nn.Module):
    def __init__(self, num_labels_task_b, num_labels_task_c,
                 demo_vec_dim=66, dropout_prob=0.1):
        super().__init__()

        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-large")
        text_dim = self.encoder.config.hidden_size  # 1024

        self.vision_encoder = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-large-patch14"
        )
        vision_dim = self.vision_encoder.config.hidden_size  # 1024
        frozen_layers = 4
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
        for layer in self.vision_encoder.encoder.layers[-frozen_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

        shared_dim = 512
        self.demo_gate = nn.Linear(demo_vec_dim, shared_dim)
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, shared_dim), nn.LayerNorm(shared_dim)
        )
        self.vision_proj = nn.Sequential(
            nn.Linear(vision_dim, shared_dim), nn.LayerNorm(shared_dim)
        )

        # ── Demographics → projected to shared_dim, ADDED to text_cls ─────────
        # GrootWatch: "embedded into the CLS token representation"
        self.demo_proj = nn.Sequential(
            nn.Linear(demo_vec_dim, shared_dim),
            nn.LayerNorm(shared_dim),
        )

        # ── Fusion: 512 (text+demo) + 512 (vision) = 1024 ────────────────────
        fused_dim = shared_dim * 2
        self.fusion_norm = nn.LayerNorm(fused_dim)
        self.dropout = nn.Dropout(dropout_prob)

        self.head_a = nn.Linear(fused_dim, 1)
        self.head_b = nn.Linear(fused_dim, num_labels_task_b)
        self.head_c = nn.Linear(fused_dim, num_labels_task_c)

        self.loss_a_fn = nn.BCEWithLogitsLoss()
        self.loss_c_fn = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, input_ids, attention_mask,
                pixel_values=None, demo_vec=None,
                labels_a=None, labels_b=None, labels_c=None, **kwargs):

        # === Text ===
        text_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = self.text_proj(text_out.last_hidden_state[:, 0, :])  # [B, 512]

        # === Inject demographics into text CLS ───────────────────────────────
        if demo_vec is not None:
            gate = torch.sigmoid(self.demo_gate(demo_vec))  # [B, 512]
            text_cls = text_cls + gate * self.demo_proj(demo_vec)
        # If no demo_vec, text_cls is unchanged

        # === Vision ===
        if pixel_values is not None:
            vision_out = self.vision_encoder(pixel_values=pixel_values)
            vision_cls = self.vision_proj(vision_out.last_hidden_state[:, 0, :])
        else:
            vision_cls = torch.zeros_like(text_cls)

        # === Fusion ===
        fused = torch.cat([text_cls, vision_cls], dim=-1)  # [B, 1024]
        fused = self.fusion_norm(fused)
        fused = self.dropout(fused)

        logits_a = self.head_a(fused)
        logits_b = self.head_b(fused)
        logits_c = self.head_c(fused)

        # === Loss === (unchanged)
        loss = None
        if labels_a is not None:
            if labels_a.dim() == 2: labels_a = labels_a[:, 1]
            mask_a = labels_a != -100.0
            if mask_a.any():
                loss_a = self.loss_a_fn(logits_a[mask_a].view(-1), labels_a[mask_a])
                loss = loss_a * 2.0 if loss is None else loss + loss_a * 2.0 

        if labels_b is not None:
            mask_b = labels_b[:, 0] != -1.0
            if mask_b.any():
                log_probs_b = F.log_softmax(logits_b[mask_b], dim=-1)
                loss_b = -(labels_b[mask_b] * log_probs_b).sum(dim=-1).mean()
                loss = loss_b if loss is None else loss + loss_b

        if labels_c is not None:
            mask_c = labels_c[:, 0] != -1.0
            if mask_c.any():
                loss_c = self.loss_c_fn(logits_c[mask_c], labels_c[mask_c]).mean()
                loss = loss_c if loss is None else loss + loss_c

        if loss is None:
            loss = logits_a.sum() * 0.0

        return (loss, logits_a, logits_b, logits_c)

In [ ]:
model = MultiHeadModel(
    num_labels_task_b=len(LABELS_2_2),
    num_labels_task_c=len(LABELS_2_3),
    demo_vec_dim=DEMO_VEC_DIM,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Frozen parameters:    {frozen_params:,}")

Total parameters:     864,203,786
Trainable parameters: 611,408,906 (70.7%)
Frozen parameters:    252,794,880


In [ ]:
tokenizer  = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

def _majority_task_a(soft):
    return 'YES' if soft[0] >= 0.5 else 'NO'

stratify_col = dataset['labels_task2_1_soft'].apply(_majority_task_a)

dataset_train, dataset_val = train_test_split(
    dataset,
    test_size=0.15,
    random_state=42,
    stratify=stratify_col,
)
dataset_train = dataset_train.reset_index(drop=True)
dataset_val = dataset_val.reset_index(drop=True)

IMAGE_DIR='./Dataset/memes/training/'

train_dataset = SexismDataset(
    dataframe=dataset_train,
    tokenizer=tokenizer,
    clip_processor=clip_processor, 
    max_length=128,
    image_dir=IMAGE_DIR,            
)

val_dataset = SexismDataset(
    dataframe=dataset_val,
    tokenizer=tokenizer,
    clip_processor=clip_processor,  
    max_length=128,
    image_dir=IMAGE_DIR,            
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)

# Model training

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=30,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=0.1,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="task_a_icmsoftnorm_soft",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    fp16=True,
    report_to="none",
    dataloader_pin_memory=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

# Explicitly save the trained model bypassing Trainer.save_model(),
# which has a known issue with custom nn.Module + Accelerate + fp16:
# the Trainer state_dict may reference stale initial parameters.



Epoch,Training Loss,Validation Loss,Task A Icm Hard,Task A Icmnorm Hard,Task A Fmeasure Hard,Task A Icmsoft Soft,Task A Icmsoftnorm Soft,Task A Crossentropy Soft,Task B Icm Hard,Task B Icmnorm Hard,Task B Icmsoft Soft,Task B Icmsoftnorm Soft,Task C Icmsoft Soft,Task C Icmsoftnorm Soft,Task C Icm Hard Default,Task C Icmnorm Hard Default,Task C Icm Hard,Task C Icmnorm Hard,Task C Thresh None,Task C Thresh Ideological Inequality,Task C Thresh Misogyny Non Sexual Violence,Task C Thresh Sexual Violence,Task C Thresh Objectification,Task C Thresh Stereotyping Dominance
1,3.249684,2.910219,-0.267164,0.355973,0.524275,-1.107227,0.325281,0.988631,-0.702270,0.246328,-1.995193,0.290011,-7.322413,0.118854,-1.539867,0.125664,-1.421993,0.154319,0.050000,0.450000,0.400000,0.300000,0.450000,0.550000
2,2.881483,2.765333,0.075439,0.540669,0.707299,-0.711376,0.387746,0.933724,-0.415957,0.349749,-1.650017,0.326340,-6.839548,0.143988,-1.481981,0.139736,-1.339431,0.174390,0.350000,0.200000,0.200000,0.200000,0.350000,0.400000
3,2.721692,2.679683,0.044302,0.523883,0.696015,-0.348348,0.445031,0.907131,-0.256007,0.407526,-1.174977,0.376337,-6.138916,0.180457,-1.193546,0.209854,-1.050833,0.244547,0.350000,0.200000,0.200000,0.150000,0.300000,0.450000
4,2.537824,2.671547,0.062124,0.533490,0.703798,-0.230066,0.463696,0.901806,-0.209320,0.424390,-1.006011,0.394120,-5.889593,0.193435,-1.015751,0.253075,-0.915963,0.277333,0.500000,0.450000,0.200000,0.200000,0.300000,0.250000
5,2.351052,2.701442,0.028703,0.515474,0.691917,-0.272060,0.457069,0.920798,-0.222601,0.419593,-1.049327,0.389561,-5.317279,0.223225,-0.903313,0.280408,-0.763283,0.314449,0.400000,0.500000,0.250000,0.250000,0.300000,0.300000
6,2.209112,2.689079,0.086837,0.546813,0.711857,-0.169611,0.473236,0.912686,-0.234326,0.415357,-0.957536,0.399222,-5.242238,0.227131,-0.803283,0.304725,-0.671955,0.336650,0.400000,0.500000,0.250000,0.200000,0.250000,0.400000
7,2.136469,2.698375,0.040652,0.521915,0.696687,-0.145559,0.477031,0.914405,-0.223746,0.419179,-0.933334,0.401769,-5.093378,0.234879,-0.697492,0.330443,-0.620624,0.349129,0.450000,0.500000,0.300000,0.250000,0.450000,0.250000
8,2.075249,2.699718,0.100968,0.554431,0.715789,-0.154059,0.475690,0.918003,-0.201289,0.427291,-0.938855,0.401188,-4.804437,0.249919,-0.695146,0.331013,-0.614428,0.350635,0.350000,0.400000,0.300000,0.250000,0.450000,0.400000
9,2.062169,2.705842,0.065264,0.535183,0.704786,-0.172297,0.472812,0.914984,-0.254686,0.408003,-1.006623,0.394056,-4.937740,0.242981,-0.675433,0.335805,-0.608144,0.352162,0.400000,0.500000,0.300000,0.250000,0.250000,0.300000
10,2.041537,2.716178,0.084961,0.545802,0.711251,-0.152897,0.475873,0.923139,-0.253936,0.408274,-0.964449,0.398494,-4.688427,0.255958,-0.676605,0.335520,-0.587473,0.357188,0.250000,0.450000,0.350000,0.250000,0.500000,0.300000


/home/a67606/jupyter-env/lib/python3.12/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


2026-05-14 09:39:50,769 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-05-14 09:39:50,815 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 09:39:50,940 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-05-14 09:39:50,942 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 09:39:51,077 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 09:39:51,204 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-05-14 09:39:51,343 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-05-14 09:39:51,445 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

/home/a67606/jupyter-env/lib/python3.12/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


2026-05-14 09:42:19,992 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-05-14 09:42:20,035 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 09:42:20,164 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-05-14 09:42:20,165 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 09:42:20,292 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 09:42:20,419 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-05-14 09:42:20,545 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-05-14 09:42:20,646 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

/home/a67606/jupyter-env/lib/python3.12/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


... (52 outputs left out)

2026-05-14 10:48:41,962 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-05-14 10:48:42,006 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 10:48:42,136 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-05-14 10:48:42,138 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 10:48:42,269 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 10:48:42,396 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-05-14 10:48:42,528 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-05-14 10:48:42,629 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

/home/a67606/jupyter-env/lib/python3.12/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


2026-05-14 10:51:10,996 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-05-14 10:51:11,040 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 10:51:11,170 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-05-14 10:51:11,171 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 10:51:11,301 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-14 10:51:11,431 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-05-14 10:51:11,559 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-05-14 10:51:11,659 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

SafetensorError: Error while serializing: I/O error: No such file or directory (os error 2)

In [ ]:
import safetensors.torch, os

from accelerate.utils import extract_model_from_parallel
actual_model = extract_model_from_parallel(trainer.model)
state_dict = {k: v.cpu().clone() for k, v in actual_model.state_dict().items()}
safetensors.torch.save_file(state_dict, "./best_model/model.safetensors")
tokenizer.save_pretrained("./best_model")
print("Model and tokenizer saved to ./best_model")

Model and tokenizer saved to ./best_model


In [ ]:
def diagnose_vision_contribution(model, val_loader, device='cuda'):
    """
    Run validation TWICE — once with real images, once with zeros.
    If results are nearly identical, vision isn't contributing.
    """
    model.eval()
    
    results = {'real_images': [], 'zero_images': []}
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            real_pixels = batch['pixel_values'].to(device)
            zero_pixels = torch.zeros_like(real_pixels)
            
            # Run with real images
            _, logits_a_real, logits_b_real, logits_c_real = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=real_pixels,
            )
            
            # Run with zeroed images
            _, logits_a_zero, logits_b_zero, logits_c_zero = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=zero_pixels,
            )
            
            # Capture probabilities for comparison
            probs_a_real = torch.sigmoid(logits_a_real).cpu().numpy()
            probs_a_zero = torch.sigmoid(logits_a_zero).cpu().numpy()
            
            results['real_images'].append(probs_a_real)
            results['zero_images'].append(probs_a_zero)
    
    real = np.concatenate(results['real_images']).flatten()
    zero = np.concatenate(results['zero_images']).flatten()
    
    # Compute how different the predictions are
    diff = np.abs(real - zero)
    correlation = np.corrcoef(real, zero)[0, 1]
    
    print("=" * 60)
    print("VISION ABLATION DIAGNOSTIC")
    print("=" * 60)
    print(f"Mean abs difference between real-image and zero-image predictions:")
    print(f"  {diff.mean():.4f}  (Task A only)")
    print(f"")
    print(f"Correlation between real and zero predictions: {correlation:.4f}")
    print(f"  - 1.00 = vision has NO effect (model ignores images)")
    print(f"  - 0.50 = moderate vision contribution")
    print(f"  - 0.00 = vision dominates / random")
    print(f"")
    
    if correlation > 0.99:
        print("⚠️  WARNING: Vision is essentially being ignored!")
    elif correlation > 0.95:
        print("⚠️  Vision contributes very little.")
    elif correlation > 0.85:
        print("⚠️  Vision contributes modestly.")
    else:
        print("✓ Vision is meaningfully contributing.")
    
    return correlation, diff.mean()

In [ ]:
diagnose_vision_contribution(actual_model, val_loader)

VISION ABLATION DIAGNOSTIC
Mean abs difference between real-image and zero-image predictions:
  0.2505  (Task A only)

Correlation between real and zero predictions: 0.4714
  - 1.00 = vision has NO effect (model ignores images)
  - 0.50 = moderate vision contribution
  - 0.00 = vision dominates / random

✓ Vision is meaningfully contributing.


(0.47137294721324813, 0.25049326)

In [ ]:
def check_clip_output_diversity(model, val_loader, device='cuda', n_samples=20):
    """
    Check that CLIP produces meaningfully different outputs for different images.
    """
    model.eval()
    cls_outputs = []
    
    with torch.no_grad():
        count = 0
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(device)
            
            # Just run the vision encoder
            vision_out = model.vision_encoder(pixel_values=pixel_values)
            cls_token = vision_out.last_hidden_state[:, 0, :]  # [batch, 1024]
            
            cls_outputs.append(cls_token.cpu().numpy())
            
            count += pixel_values.shape[0]
            if count >= n_samples:
                break
    
    cls_outputs = np.concatenate(cls_outputs, axis=0)[:n_samples]
    
    # Pairwise cosine similarities
    from numpy.linalg import norm
    similarities = []
    for i in range(len(cls_outputs)):
        for j in range(i + 1, len(cls_outputs)):
            sim = np.dot(cls_outputs[i], cls_outputs[j]) / (
                norm(cls_outputs[i]) * norm(cls_outputs[j]) + 1e-8
            )
            similarities.append(sim)
    
    similarities = np.array(similarities)
    
    print("=" * 60)
    print("CLIP OUTPUT DIVERSITY CHECK")
    print("=" * 60)
    print(f"Pairwise cosine similarities between {n_samples} CLIP outputs:")
    print(f"  Mean: {similarities.mean():.4f}")
    print(f"  Std:  {similarities.std():.4f}")
    print(f"  Min:  {similarities.min():.4f}")
    print(f"  Max:  {similarities.max():.4f}")
    print()
    
    if similarities.mean() > 0.99:
        print("⚠️  CLIP outputs are nearly identical for all images!")
        print("   This means CLIP isn't seeing diverse content.")
        print("   Possible causes: all images are the black fallback")
        print("                    or preprocessing is broken")
    elif similarities.mean() > 0.95:
        print("⚠️  CLIP outputs are very similar (low diversity).")
    else:
        print("✓ CLIP is producing diverse outputs.")
    
    return similarities
check_clip_output_diversity(actual_model, val_loader)

CLIP OUTPUT DIVERSITY CHECK
Pairwise cosine similarities between 20 CLIP outputs:
  Mean: 0.1765
  Std:  0.0711
  Min:  0.0031
  Max:  0.3730

✓ CLIP is producing diverse outputs.


array([0.13593686, 0.1957589 , 0.15229633, 0.20919472, 0.14253775,
       0.19869089, 0.15901117, 0.22103789, 0.18482567, 0.24985182,
       0.15674052, 0.22981192, 0.16440657, 0.23710665, 0.22002319,
       0.24956516, 0.10316042, 0.09670394, 0.20996932, 0.23860578,
       0.12741326, 0.22793791, 0.12563594, 0.15593788, 0.10879819,
       0.18582984, 0.14103696, 0.13615884, 0.1748575 , 0.05332811,
       0.14473004, 0.12861975, 0.10805496, 0.00307232, 0.10371892,
       0.16815191, 0.23142047, 0.10854193, 0.13187857, 0.13218499,
       0.11228061, 0.14053753, 0.2910953 , 0.30909265, 0.1209649 ,
       0.37302585, 0.27998483, 0.18111688, 0.25144828, 0.1909319 ,
       0.1445615 , 0.04894502, 0.03176033, 0.27893232, 0.15472264,
       0.17776697, 0.26861536, 0.19827643, 0.14556674, 0.20071505,
       0.21812554, 0.12259271, 0.23421633, 0.20295288, 0.07544491,
       0.17829308, 0.23064929, 0.09021128, 0.18748592, 0.11733566,
       0.08044151, 0.2233735 , 0.13397767, 0.13863968, 0.15215